# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [43]:
!cat publications.tsv

pub_date	title	venue	excerpt	citation	url_slug	paper_url
2022-10-01	Editorial (Português): Uma breve história do hardware, seus desafios, e impacto sobre o consumo de energia	Academic Journal on Computing, Engineering and Applied Mathematics	Este artigo representa a opinião do autor sobre o campo da arquitetura de computadores e os desafios futuros. Este trabalho tenta estabelecer como a computação se tornou tão importante como é hoje. Além disso, procura elencar alguns dos principais desafios da área de projetos de computadores. Assim, este artigo foi escrito para estudantes e pesquisadores que desejam ver o panorama geral do assunto, e nenhum destes é amplamente discutido, depois, essa tarefa é quase impossível e não é a intenção. No entanto, alguns grandes insights podem surgir deste artigo, uma vez que o futuro parece ser co-design e aproximado.	Almeida, T. da S. 2022. Editorial (Português): Uma breve história do hardware, seus desafios, e impacto sobre o consumo de energia. Academ

## Import pandas

We are using the very handy pandas library for dataframes.

In [44]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [45]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0)
publications


,pub_date,title,venue,excerpt,citation,url_slug,paper_url
0,2022-10-01,Editorial (Português): Uma breve história do h...,"Academic Journal on Computing, Engineering and...",Este artigo representa a opinião do autor sobr...,"Almeida, T. da S. 2022. Editorial (Português):...",ajceam-5,NaN
1,2021-12-01,Optimization of state assignment in a finite s...,"Academic Journal on Computing, Engineering and...","In this research, the application of the Simul...","da Silva Ribeiro, R., Lima de Carvalho, R. e d...",ajceam-4,https://almeidatiago.gisthub.io/files/paper1.pdf
2,2021-04-01,Evaluation of a Sliding Window mechanism as Da...,"Academic Journal on Computing, Engineering and...",Emotion analysis is an important field of stud...,"Farias da Silva, M.A., de Carvalho, R.L. e Alm...",ajceam-3,NaN
3,2020-06-01,Evaluation of Clause-Column Table method to de...,"Academic Journal on Computing, Engineering and...",Asynchronous finite state machines are of grea...,"Almeida, T. da S. e Luiz Gomes de Freitas, A. ...",ajceam-2,NaN
4,2020-03-01,Technical report of Raspberry Pi prototype for...,"Academic Journal on Computing, Engineering and...",This paper presents a low-cost prototype for l...,"Almeida, T. da S. 2020. Technical report of Ra...",ajceam-1,NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [46]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [47]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    #md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"
    
    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [48]:
!ls ../_publications/

2020-03-01-ajceam-1.md	2021-04-01-ajceam-3.md	2022-10-01-ajceam-5.md
2020-06-01-ajceam-2.md	2021-12-01-ajceam-4.md


In [49]:
!cat ../_publications/2020-03-01-ajceam-1.md

---
title: "Technical report of Raspberry Pi prototype for lectures in public universities"
collection: publications
permalink: /publication/2020-03-01-ajceam-1
excerpt: 'This paper presents a low-cost prototype for lectures in public universities based on Raspberry Pi. It is described how to connect and configure the prototype using an infrared remote control. Technologies applied in education are widely explored in recent researches and can be held to lower the cost. The proposed prototype is 86\% cheaper on average (compared to ordinary computer) and can be used for automation of the classrooms besides lectures. For example, the prototype can be additionally used to access control, environment monitoring, and management of the environment utilization by the users.'
date: 2020-03-01
venue: 'Academic Journal on Computing, Engineering and Applied Mathematics'
---
This paper presents a low-cost prototype for lectures in public universities based on Raspberry Pi. It is described how to c